# Demonstrating State-Level Variable Changes in PolicyEngine Microsimulations

## The Problem

When you change an input variable (like `state_fips`) in a loaded microsimulation, PolicyEngine does **NOT** automatically invalidate cached calculated variables that depend on it. This means derived variables (like `state_income_tax`) will return stale values even after you change the state.

## The Solution

After changing an input, you must manually delete ALL calculated variable caches to force fresh recalculation.

## The Example

We test household 79855 which lives in California and pays $2,939 in state income tax. If we move them to Texas (which has no income tax), their `state_income_tax` should drop to $0. Without clearing caches, it would incorrectly remain at $2,939.

## Setup

In [1]:
import numpy as np
from policyengine_us import Microsimulation

# Configuration
TEST_HOUSEHOLD_ID = 79855  # CA household paying state income tax
CALIFORNIA_FIPS = 6
TEXAS_FIPS = 48
YEAR = 2023

/home/baogorek/envs/pe/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load Microsimulation and Verify Baseline

First, we load the CPS 2023 microdata and check the baseline state income tax for our test household in California.

In [2]:
print("Loading CPS 2023 microdata...")
sim = Microsimulation(dataset="hf://policyengine/policyengine-us-data/cps_2023.h5")

# Calculate baseline for our test household
baseline = sim.calculate_dataframe(
    ['household_id', 'state_fips', 'state_code_str',
     'adjusted_gross_income', 'state_income_tax'],
    map_to='household'
)

test_hh_baseline = baseline[baseline['household_id'] == TEST_HOUSEHOLD_ID].iloc[0]

print(f"\nBaseline for Household {TEST_HOUSEHOLD_ID}:")
print(f"  State: {test_hh_baseline['state_code_str']} (FIPS: {int(test_hh_baseline['state_fips'])})")
print(f"  AGI: ${test_hh_baseline['adjusted_gross_income']:,.2f}")
print(f"  State Income Tax: ${test_hh_baseline['state_income_tax']:,.2f}")

ca_tax = test_hh_baseline['state_income_tax']

Loading CPS 2023 microdata...

Baseline for Household 79855:
  State: CA (FIPS: 6)
  AGI: $74,665.20
  State Income Tax: $2,938.78


## Step 2: Change State to Texas

Now we modify the `state_fips` for our test household only, changing it from California (6) to Texas (48). All other households remain in their original states.

In [3]:
print(f"Changing household {TEST_HOUSEHOLD_ID} from CA to TX...")

# Get all household IDs and current states
all_household_ids = sim.calculate('household_id', YEAR)
original_states = sim.calculate('state_fips', YEAR)

# Create a modified state array where only our test household changes
# All other households keep their original state
modified_states = np.where(
    all_household_ids == TEST_HOUSEHOLD_ID,
    TEXAS_FIPS,  # Change this household to Texas
    original_states  # Keep everyone else the same
)

# Override the state_fips input
# NOTE: set_input automatically overwrites the microdata value - no need to delete first
sim.set_input('state_fips', YEAR, modified_states)

print(f"  Changed state_fips for household {TEST_HOUSEHOLD_ID}: {CALIFORNIA_FIPS} → {TEXAS_FIPS}")

Changing household 79855 from CA to TX...
  Changed state_fips for household 79855: 6 → 48


## Step 3: Nuclear Delete - Clear All Calculated Variable Caches

This is the **critical step**. PolicyEngine doesn't automatically invalidate caches when inputs change, so we must manually delete all calculated variable caches.

We delete only **calculated** variables (derived from inputs), NOT input variables (which contain the actual microdata like employment_income, age, etc.).

In [4]:
print("Clearing all calculated variable caches...")
print("(This is necessary because PolicyEngine doesn't auto-invalidate caches)")

# Get list of all variables and input variables
all_variables = sim.tax_benefit_system.variables
input_variables = sim.input_variables

deleted_count = 0

# Delete all CALCULATED variables (anything not in input_variables)
for variable_name in all_variables:
    if variable_name not in input_variables:
        try:
            sim.delete_arrays(variable_name, YEAR)
            deleted_count += 1
        except:
            # Some variables may not have arrays - that's fine
            pass

print(f"  Deleted {deleted_count} calculated variable caches")

Clearing all calculated variable caches...
(This is necessary because PolicyEngine doesn't auto-invalidate caches)
  Deleted 3261 calculated variable caches


## Step 4: Recalculate and Verify

Now we recalculate with fresh caches. The state income tax should now reflect Texas's tax policy (which is $0).

In [5]:
print("Recalculating with fresh caches...")

result = sim.calculate_dataframe(
    ['household_id', 'state_fips', 'state_code_str',
     'adjusted_gross_income', 'state_income_tax'],
    map_to='household'
)

test_hh_result = result[result['household_id'] == TEST_HOUSEHOLD_ID].iloc[0]

print(f"\nResult for Household {TEST_HOUSEHOLD_ID} after state swap:")
print(f"  State: {test_hh_result['state_code_str']} (FIPS: {int(test_hh_result['state_fips'])})")
print(f"  AGI: ${test_hh_result['adjusted_gross_income']:,.2f}")
print(f"  State Income Tax: ${test_hh_result['state_income_tax']:,.2f}")

tx_tax = test_hh_result['state_income_tax']

Recalculating with fresh caches...

Result for Household 79855 after state swap:
  State: TX (FIPS: 48)
  AGI: $74,665.20
  State Income Tax: $0.00


## Verification

Let's verify that the state swap worked correctly:

In [6]:
# Check that state changed
state_changed = test_hh_result['state_fips'] == TEXAS_FIPS
print(f"✓ State successfully changed: {state_changed}")

# Check that tax went to zero (Texas has no income tax)
tax_zeroed = tx_tax == 0.0
print(f"✓ State income tax correctly went to $0: {tax_zeroed}")

# Calculate the tax savings from moving
tax_savings = ca_tax - tx_tax
print(f"\nTax savings from CA → TX: ${tax_savings:,.2f}")

if state_changed and tax_zeroed:
    print("\n✅ SUCCESS! State swap worked correctly with cache invalidation.")
else:
    print("\n❌ FAILURE! Something went wrong with the state swap.")

✓ State successfully changed: True
✓ State income tax correctly went to $0: True

Tax savings from CA → TX: $2,938.78

✅ SUCCESS! State swap worked correctly with cache invalidation.


## Key Takeaway

After changing **ANY** input variable in a PolicyEngine microsimulation:

1. **Use `set_input()`** to change the input value
2. **Delete ALL calculated variable caches** (the "nuclear option")
3. **Recalculate** - values will be fresh and correct

Without step 2, calculated variables will return stale cached values!

### The Nuclear Delete Pattern

```python
# After changing any input:
for variable_name in sim.tax_benefit_system.variables:
    if variable_name not in sim.input_variables:
        try:
            sim.delete_arrays(variable_name, year)
        except:
            pass
```

This ensures all calculated values are recomputed based on your new inputs!